# Load All Dataset Files

In [ ]:
import pandas as pd
import numpy as np
import unicodedata

# Dataset Preprocessing

## olist_geolocation_dataset.csv

What anomalies were handled here

For olist_geolocation_dataset.csv, we handled:
1. exact duplicate rows → removed
2. zip prefix formatting issues → standardized
3. city text inconsistency → normalized
4. accent inconsistency → cleaned
5. state formatting inconsistency → standardized
6. invalid lat/lng types → coerced to numeric
7. missing critical location fields → removed
8. impossible coordinates → filtered
9. duplicate zip prefixes → not deleted blindly, aggregated safely


In [ ]:
# Load
geo = pd.read_csv("olist_geolocation_dataset.csv")

# Standardize column names
geo.columns = [col.strip().lower() for col in geo.columns]

# Remove exact duplicate rows
geo = geo.drop_duplicates().copy()

# Standardize zip code prefix
geo["geolocation_zip_code_prefix"] = (
    geo["geolocation_zip_code_prefix"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

# Clean city names
geo["geolocation_city"] = (
    geo["geolocation_city"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

def remove_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', str(text))
        if not unicodedata.combining(c)
    )

geo["geolocation_city_clean"] = geo["geolocation_city"].apply(remove_accents)

# Standardize state
geo["geolocation_state"] = (
    geo["geolocation_state"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Convert coordinates
geo["geolocation_lat"] = pd.to_numeric(geo["geolocation_lat"], errors="coerce")
geo["geolocation_lng"] = pd.to_numeric(geo["geolocation_lng"], errors="coerce")

# Drop rows missing critical fields
geo = geo.dropna(subset=[
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_state"
]).copy()

# Filter invalid coordinates
geo = geo[
    geo["geolocation_lat"].between(-34, 6) &
    geo["geolocation_lng"].between(-74, -28)
].copy()

# Aggregation helper
def safe_mode(series):
    mode = series.mode()
    if len(mode) > 0:
        return mode.iloc[0]
    return series.iloc[0]

# Create zip-level geolocation table
geo_zip = geo.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    geolocation_lat=("geolocation_lat", "median"),
    geolocation_lng=("geolocation_lng", "median"),
    geolocation_city=("geolocation_city_clean", safe_mode),
    geolocation_state=("geolocation_state", safe_mode),
    geolocation_record_count=("geolocation_zip_code_prefix", "size")
)

# Validation
print("Clean raw geolocation shape:", geo.shape)
print("Zip-level geolocation shape:", geo_zip.shape)
print("Exact duplicates remaining:", geo.duplicated().sum())
print("Duplicate zip prefixes in zip table:", geo_zip["geolocation_zip_code_prefix"].duplicated().sum())

# Save
geo.to_csv("geolocation_clean.csv", index=False)
geo_zip.to_csv("geolocation_zip_level.csv", index=False)

Clean raw geolocation shape: (738305, 6)
Zip-level geolocation shape: (19011, 6)
Exact duplicates remaining: 0
Duplicate zip prefixes in zip table: 0


## olist_products_dataset.csv & product_category_name_translation

What anomalies were handled here

For olist_products_dataset.csv, we handled:
1. exact duplicate rows → removed
2. column naming inconsistency (lenght) → fixed
3. category text inconsistency → standardized
4. missing category → filled with "unknown"
5. missing metadata fields → imputed
6. missing physical attributes → imputed
7. zero / invalid physical values → converted to missing then imputed
8. category translation mismatch → filled untranslated categories as "unknown"
9. new derived logistics features → created

In [ ]:
import pandas as pd
import numpy as np

# Load products
products = pd.read_csv("olist_products_dataset.csv")

# Standardize column names
products.columns = [col.strip().lower() for col in products.columns]
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

# Remove exact duplicates
products = products.drop_duplicates().copy()

# Clean category text
products["product_category_name"] = (
    products["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Fill missing category
products["product_category_name"] = products["product_category_name"].fillna("unknown")

# Numeric conversion
numeric_cols = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in numeric_cols:
    products[col] = pd.to_numeric(products[col], errors="coerce")

# Metadata columns
text_meta_cols = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty"
]

# Physical columns
physical_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

# Replace invalid physical values with NaN
for col in physical_cols:
    products.loc[products[col] <= 0, col] = np.nan

# Impute text metadata by category median, then global median
for col in text_meta_cols:
    products[col] = products.groupby("product_category_name")[col].transform(
        lambda x: x.fillna(x.median())
    )
    products[col] = products[col].fillna(products[col].median())

# Impute physical attributes by category median, then global median
for col in physical_cols:
    products[col] = products.groupby("product_category_name")[col].transform(
        lambda x: x.fillna(x.median())
    )
    products[col] = products[col].fillna(products[col].median())

# Derived features
products["product_volume_cm3"] = (
    products["product_length_cm"] *
    products["product_height_cm"] *
    products["product_width_cm"]
)

products["product_density_g_cm3"] = np.where(
    products["product_volume_cm3"] > 0,
    products["product_weight_g"] / products["product_volume_cm3"],
    np.nan
)

products["flag_unknown_category"] = (
    products["product_category_name"] == "unknown"
).astype(int)

# Optional category translation
category_tr = pd.read_csv("product_category_name_translation.csv")
category_tr.columns = [col.strip().lower() for col in category_tr.columns]

category_tr["product_category_name"] = (
    category_tr["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

category_tr["product_category_name_english"] = (
    category_tr["product_category_name_english"]
    .astype("string")
    .str.strip()
    .str.lower()
)

products = products.merge(
    category_tr,
    on="product_category_name",
    how="left"
)

products["product_category_name_english"] = (
    products["product_category_name_english"].fillna("unknown")
)

# Validation
print("Final shape:", products.shape)
print("Duplicate product_id:", products["product_id"].duplicated().sum())
print(products.isna().sum())

# Save
products.to_csv("products_clean.csv", index=False)

Final shape: (32951, 13)
Duplicate product_id: 0
product_id                       0
product_category_name            0
product_name_length              0
product_description_length       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_volume_cm3               0
product_density_g_cm3            0
flag_unknown_category            0
product_category_name_english    0
dtype: int64


## olist_customers_dataset.csv

What anomalies were handled

For olist_customers_dataset.csv, we handled:
1. exact duplicates → removed
2. zip code formatting issues → standardized
3. city text inconsistency → normalized
4. accent inconsistency → cleaned
5. state formatting → standardized
6. invalid state detection → flagged
7. multiple customer_id per real customer → correctly handled
8. customer-level aggregation → created

In [ ]:
import pandas as pd
import unicodedata

# Load
customers = pd.read_csv("olist_customers_dataset.csv")

# Standardize columns
customers.columns = [col.strip().lower() for col in customers.columns]

# Remove duplicates
customers = customers.drop_duplicates().copy()

# Fix zip code
customers["customer_zip_code_prefix"] = (
    customers["customer_zip_code_prefix"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

# Clean city
customers["customer_city"] = (
    customers["customer_city"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

def remove_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', str(text))
        if not unicodedata.combining(c)
    )

customers["customer_city_clean"] = customers["customer_city"].apply(remove_accents)

# Clean state
customers["customer_state"] = (
    customers["customer_state"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# State validation
customers["flag_invalid_state"] = customers["customer_state"].str.len() != 2

# Customer order count
customer_order_count = (
    customers.groupby("customer_unique_id")
    .size()
    .reset_index(name="customer_order_count")
)

customers = customers.merge(
    customer_order_count,
    on="customer_unique_id",
    how="left"
)

# Flags
customers["is_repeat_customer"] = (
    customers["customer_order_count"] > 1
).astype(int)

customers["is_new_customer"] = (
    customers["customer_order_count"] == 1
).astype(int)

# Customer master table
customer_master = (
    customers.groupby("customer_unique_id", as_index=False)
    .agg({
        "customer_zip_code_prefix": "first",
        "customer_city_clean": "first",
        "customer_state": "first",
        "customer_order_count": "max"
    })
)

customer_master["is_repeat_customer"] = (
    customer_master["customer_order_count"] > 1
).astype(int)

# Save
customers.to_csv("customers_clean.csv", index=False)
customer_master.to_csv("customer_master.csv", index=False)

## olist_sellers_dataset.csv

What anomalies were handled

For olist_sellers_dataset.csv, we handled:
1. exact duplicates → removed
2. zip code formatting issues → standardized
3. city text inconsistency → normalized
4. accent inconsistency → cleaned
5. state formatting issues → standardized
6. invalid state codes → flagged
7. zip vs city/state inconsistency → audited (not blindly removed)
8. region grouping → derived feature

In [ ]:
import pandas as pd
import unicodedata

# Load
sellers = pd.read_csv("olist_sellers_dataset.csv")

# Standardize column names
sellers.columns = [col.strip().lower() for col in sellers.columns]

# Remove duplicates
sellers = sellers.drop_duplicates().copy()

# Fix zip code
sellers["seller_zip_code_prefix"] = (
    sellers["seller_zip_code_prefix"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

# Clean city
sellers["seller_city"] = (
    sellers["seller_city"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Remove accents
def remove_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', str(text))
        if not unicodedata.combining(c)
    )

sellers["seller_city_clean"] = sellers["seller_city"].apply(remove_accents)

# Clean state
sellers["seller_state"] = (
    sellers["seller_state"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Validate state
sellers["flag_invalid_state"] = sellers["seller_state"].str.len() != 2

# Region mapping
region_map = {
    "SP": "southeast", "RJ": "southeast", "MG": "southeast", "ES": "southeast",
    "RS": "south", "SC": "south", "PR": "south",
    "BA": "northeast", "PE": "northeast", "CE": "northeast",
    "DF": "center", "GO": "center", "MT": "center", "MS": "center",
    "AM": "north", "PA": "north"
}

sellers["seller_region"] = sellers["seller_state"].map(region_map)
sellers["seller_region"] = sellers["seller_region"].fillna("other")

# Flag missing geo info
sellers["flag_missing_geo_info"] = (
    sellers["seller_city"].isna() |
    sellers["seller_state"].isna()
).astype(int)

# Validation
print("Final shape:", sellers.shape)
print("Duplicate seller_id:", sellers["seller_id"].duplicated().sum())
print("Missing values:\n", sellers.isna().sum())

# Save
sellers.to_csv("sellers_clean.csv", index=False)

Final shape: (3095, 8)
Duplicate seller_id: 0
Missing values:
 seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
seller_city_clean         0
flag_invalid_state        0
seller_region             0
flag_missing_geo_info     0
dtype: int64


## olist_orders_dataset.csv

What anomalies were handled

For olist_orders_dataset.csv, we handled:
1. missing timestamps → preserved + interpreted
2. chronology anomalies → flagged
3. invalid timeline sequences → flagged (not removed)
4. status inconsistency → detected
5. lifecycle interpretation → engineered
6. time features → created
7. data quality indicators → added

In [ ]:
import pandas as pd
import numpy as np

# Load
orders = pd.read_csv("olist_orders_dataset.csv")

# Standardize columns
orders.columns = [col.strip().lower() for col in orders.columns]

# Remove duplicates
orders = orders.drop_duplicates().copy()

# Convert dates
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Clean status
orders["order_status"] = (
    orders["order_status"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Lifecycle flags
orders["is_approved"] = orders["order_approved_at"].notna().astype(int)
orders["is_shipped"] = orders["order_delivered_carrier_date"].notna().astype(int)
orders["is_delivered"] = orders["order_delivered_customer_date"].notna().astype(int)

# Lifecycle stage
def get_stage(row):
    if pd.notna(row["order_delivered_customer_date"]):
        return "delivered"
    elif pd.notna(row["order_delivered_carrier_date"]):
        return "shipped"
    elif pd.notna(row["order_approved_at"]):
        return "approved"
    else:
        return "created"

orders["order_stage"] = orders.apply(get_stage, axis=1)

# Anomaly flags
orders["flag_carrier_before_approval"] = (
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
).fillna(False)

orders["flag_customer_before_carrier"] = (
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
).fillna(False)

# Time features
orders["approval_delay_hours"] = (
    (orders["order_approved_at"] - orders["order_purchase_timestamp"])
    .dt.total_seconds() / 3600
)

orders["shipping_delay_days"] = (
    (orders["order_delivered_carrier_date"] - orders["order_approved_at"])
    .dt.total_seconds() / 86400
)

orders["delivery_days"] = (
    (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"])
    .dt.total_seconds() / 86400
)

orders["estimated_delivery_days"] = (
    (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"])
    .dt.total_seconds() / 86400
)

orders["delivery_delay_days"] = (
    (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"])
    .dt.total_seconds() / 86400
)

orders["is_late_delivery"] = (orders["delivery_delay_days"] > 0).astype(float)

# Date features
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_day"] = orders["order_purchase_timestamp"].dt.day
orders["purchase_hour"] = orders["order_purchase_timestamp"].dt.hour
orders["purchase_weekday"] = orders["order_purchase_timestamp"].dt.day_name()

# Data quality flags
orders["flag_inconsistent_status_delivery"] = (
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
).astype(int)

orders["flag_canceled_but_delivered"] = (
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_customer_date"].notna())
).astype(int)

# Save
orders.to_csv("orders_clean.csv", index=False)

## olist_order_items_dataset.csv

What anomalies were handled

For olist_order_items_dataset.csv, we handled:
1. exact duplicates → removed
2. composite key validation → ensured uniqueness
3. numeric conversion issues → fixed
4. negative price/freight → corrected
5. zero price → flagged
6. outliers → detected
7. high freight anomaly → flagged
8. order-level aggregation → created

In [ ]:
import pandas as pd
import numpy as np

# Load
items = pd.read_csv("olist_order_items_dataset.csv")

# Standardize columns
items.columns = [col.strip().lower() for col in items.columns]

# Remove duplicates
items = items.drop_duplicates().copy()

# Validate composite key
assert items.duplicated(subset=["order_id", "order_item_id"]).sum() == 0

# Convert datetime
items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"], errors="coerce"
)

# Convert numeric
items["price"] = pd.to_numeric(items["price"], errors="coerce")
items["freight_value"] = pd.to_numeric(items["freight_value"], errors="coerce")

# Handle negative values
items.loc[items["price"] < 0, "price"] = np.nan
items.loc[items["freight_value"] < 0, "freight_value"] = np.nan

# Flags
items["flag_zero_price"] = (items["price"] == 0).astype(int)

upper_bound = items["price"].quantile(0.99)
items["flag_price_outlier"] = (items["price"] > upper_bound).astype(int)

# Derived features
items["total_item_value"] = items["price"] + items["freight_value"]

items["freight_ratio"] = np.where(
    items["price"] > 0,
    items["freight_value"] / items["price"],
    np.nan
)

items["flag_high_freight"] = (items["freight_ratio"] > 1).astype(int)

# Aggregation
items_agg = items.groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "count"),
    item_price_total=("price", "sum"),
    item_price_mean=("price", "mean"),
    item_price_max=("price", "max"),
    freight_total=("freight_value", "sum"),
    freight_mean=("freight_value", "mean"),
    seller_count=("seller_id", "nunique"),
    product_count=("product_id", "nunique")
)

items_agg["order_gross_value"] = (
    items_agg["item_price_total"] + items_agg["freight_total"]
)

# Save
items.to_csv("order_items_clean.csv", index=False)
items_agg.to_csv("order_items_agg.csv", index=False)

## olist_order_payments_dataset.csv

What anomalies were handled

For olist_order_payments_dataset.csv, we handled:
1. exact duplicates → removed
2. multi-row per order → properly aggregated
3. negative payment values → corrected
4. zero payment values → flagged
5. high installments → flagged
6. payment type inconsistency → normalized
7. multiple payment types per order → captured
8. payment composition → pivoted

In [ ]:
import pandas as pd
import numpy as np

# Load
payments = pd.read_csv("olist_order_payments_dataset.csv")

# Standardize columns
payments.columns = [col.strip().lower() for col in payments.columns]

# Remove duplicates
payments = payments.drop_duplicates().copy()

# Convert numeric
payments["payment_sequential"] = pd.to_numeric(payments["payment_sequential"], errors="coerce")
payments["payment_installments"] = pd.to_numeric(payments["payment_installments"], errors="coerce")
payments["payment_value"] = pd.to_numeric(payments["payment_value"], errors="coerce")

# Clean payment type
payments["payment_type"] = (
    payments["payment_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Handle negative values
payments.loc[payments["payment_value"] < 0, "payment_value"] = np.nan

# Flags
payments["flag_zero_payment"] = (payments["payment_value"] == 0).astype(int)
payments["flag_high_installments"] = (
    payments["payment_installments"] > 12
).astype(int)

# Aggregation
payments_agg = payments.groupby("order_id", as_index=False).agg(
    payment_value_total=("payment_value", "sum"),
    payment_value_mean=("payment_value", "mean"),
    payment_value_max=("payment_value", "max"),
    payment_installments_max=("payment_installments", "max"),
    payment_installments_mean=("payment_installments", "mean"),
    payment_count=("payment_sequential", "count")
)

# Mode function
def safe_mode(series):
    mode = series.mode()
    if len(mode) > 0:
        return mode.iloc[0]
    return series.iloc[0]

# Main payment type
payment_type_main = (
    payments.groupby("order_id")["payment_type"]
    .agg(safe_mode)
    .reset_index(name="payment_type_main")
)

# Unique payment types
payment_type_nunique = (
    payments.groupby("order_id")["payment_type"]
    .nunique()
    .reset_index(name="payment_type_nunique")
)

# Merge
payments_agg = payments_agg.merge(payment_type_main, on="order_id", how="left")
payments_agg = payments_agg.merge(payment_type_nunique, on="order_id", how="left")

# Multi-payment flag
payments_agg["flag_multi_payment"] = (
    payments_agg["payment_count"] > 1
).astype(int)

# Payment type pivot
payment_type_pivot = (
    payments.assign(count=1)
    .pivot_table(
        index="order_id",
        columns="payment_type",
        values="count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

payment_type_pivot.columns = [
    "order_id" if col == "order_id" else f"payment_{col}_count"
    for col in payment_type_pivot.columns
]

payments_agg = payments_agg.merge(payment_type_pivot, on="order_id", how="left")

# Save
payments.to_csv("payments_clean.csv", index=False)
payments_agg.to_csv("payments_order_level.csv", index=False)

## olist_order_reviews_dataset.csv

What anomalies were handled

For olist_order_reviews_dataset.csv, we handled:
1. exact duplicate rows → removed
2. duplicate review_id → deduplicated (latest kept)
3. multiple reviews per order → handled properly
4. missing text fields → filled safely
5. text inconsistency → normalized
6. invalid review scores → flagged
7. review length variation → captured
8. sentiment classification → created

In [ ]:
import pandas as pd
import numpy as np

# Load
reviews = pd.read_csv("olist_order_reviews_dataset.csv")

# Standardize columns
reviews.columns = [col.strip().lower() for col in reviews.columns]

# Remove exact duplicates
reviews = reviews.drop_duplicates().copy()

# Convert datetime
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"], errors="coerce"
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"], errors="coerce"
)

# Remove duplicate review_id (keep latest)
reviews = reviews.sort_values("review_answer_timestamp")
reviews = reviews.drop_duplicates(subset=["review_id"], keep="last")

# Clean score
reviews["review_score"] = pd.to_numeric(reviews["review_score"], errors="coerce")

# Flags
reviews["flag_invalid_score"] = ~reviews["review_score"].between(1, 5)

# Clean text
reviews["review_comment_title"] = reviews["review_comment_title"].fillna("")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("")

reviews["review_comment_message"] = (
    reviews["review_comment_message"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Features
reviews["has_comment"] = (reviews["review_comment_message"].str.len() > 0).astype(int)
reviews["review_length"] = reviews["review_comment_message"].str.len()
reviews["flag_long_review"] = (reviews["review_length"] > 100).astype(int)

# Sentiment
reviews["review_sentiment"] = reviews["review_score"].apply(
    lambda x: "positive" if x >= 4 else (
        "neutral" if x == 3 else "negative"
    )
)

reviews["is_satisfied"] = (reviews["review_score"] >= 4).astype(int)

# Order-level dataset (latest review)
reviews_order = (
    reviews.sort_values("review_answer_timestamp")
    .drop_duplicates(subset=["order_id"], keep="last")
)

# Aggregated dataset
reviews_agg = reviews.groupby("order_id", as_index=False).agg(
    review_score_mean=("review_score", "mean"),
    review_score_min=("review_score", "min"),
    review_score_max=("review_score", "max"),
    review_count=("review_id", "count"),
    has_comment=("has_comment", "max"),
    review_length=("review_length", "mean")
)

# Save
reviews.to_csv("reviews_clean.csv", index=False)
reviews_order.to_csv("reviews_order_level.csv", index=False)
reviews_agg.to_csv("reviews_agg.csv", index=False)

# Generate Fact & Dimension Tables

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================
INPUT_DIR = Path(".")      # change if needed
OUTPUT_DIR = Path("./star_schema_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD CLEANED FILES
# =========================================================
# =========================================================
# LOAD CLEANED FILES
# =========================================================
orders = pd.read_csv(INPUT_DIR / "orders_clean.csv", low_memory=False)
customers = pd.read_csv(INPUT_DIR / "customers_clean.csv", low_memory=False)
products = pd.read_csv(INPUT_DIR / "products_clean.csv", low_memory=False)
sellers = pd.read_csv(INPUT_DIR / "sellers_clean.csv", low_memory=False)
items = pd.read_csv(INPUT_DIR / "order_items_clean.csv", low_memory=False)
items_agg = pd.read_csv(INPUT_DIR / "order_items_agg.csv", low_memory=False)
payments = pd.read_csv(INPUT_DIR / "payments_order_level.csv", low_memory=False)
reviews = pd.read_csv(INPUT_DIR / "reviews_order_level.csv", low_memory=False)
geo = pd.read_csv(INPUT_DIR / "geolocation_zip_level.csv", low_memory=False)

def safe_datetime(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

orders = safe_datetime(orders, [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
])

reviews = safe_datetime(reviews, [
    "review_creation_date",
    "review_answer_timestamp"
])

items = safe_datetime(items, [
    "shipping_limit_date"
])

# =========================================================
# OPTIONAL: STANDARDIZE KEY COLUMNS AS STRING
# =========================================================
key_cols_by_df = {
    "orders": (orders, ["order_id", "customer_id"]),
    "customers": (customers, ["customer_id", "customer_unique_id", "customer_zip_code_prefix"]),
    "products": (products, ["product_id"]),
    "sellers": (sellers, ["seller_id", "seller_zip_code_prefix"]),
    "items": (items, ["order_id", "product_id", "seller_id"]),
    "items_agg": (items_agg, ["order_id"]),
    "payments": (payments, ["order_id"]),
    "reviews": (reviews, ["order_id", "review_id"] if "review_id" in reviews.columns else ["order_id"]),
    "geo": (geo, ["geolocation_zip_code_prefix"]),
}

for _, (df, cols) in key_cols_by_df.items():
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str)

# =========================================================
# BUILD DIMENSION TABLES
# =========================================================

# -------------------------
# dim_customers
# Grain: 1 row = 1 customer_id
# -------------------------
dim_customers_cols = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
]

optional_customer_cols = [
    "customer_city_clean",
    "customer_lat",
    "customer_lng",
    "customer_geo_city",
    "customer_geo_state",
    "customer_geo_rows",
    "customer_order_count",
    "is_repeat_customer",
    "is_new_customer",
    "flag_invalid_state",
]

dim_customers = customers[[c for c in dim_customers_cols + optional_customer_cols if c in customers.columns]].drop_duplicates().copy()

# surrogate key
dim_customers = dim_customers.reset_index(drop=True)
dim_customers.insert(0, "customer_key", dim_customers.index + 1)

# -------------------------
# dim_products
# Grain: 1 row = 1 product_id
# -------------------------
dim_products_cols = [
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

optional_product_cols = [
    "product_volume_cm3",
    "product_density_g_cm3",
    "flag_unknown_category",
]

dim_products = products[[c for c in dim_products_cols + optional_product_cols if c in products.columns]].drop_duplicates().copy()
dim_products = dim_products.reset_index(drop=True)
dim_products.insert(0, "product_key", dim_products.index + 1)

# -------------------------
# dim_sellers
# Grain: 1 row = 1 seller_id
# -------------------------
dim_sellers_cols = [
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
]

optional_seller_cols = [
    "seller_city_clean",
    "seller_region",
    "flag_invalid_state",
    "flag_missing_geo_info",
    "seller_lat",
    "seller_lng",
    "seller_geo_city",
    "seller_geo_state",
    "seller_geo_rows",
]

dim_sellers = sellers[[c for c in dim_sellers_cols + optional_seller_cols if c in sellers.columns]].drop_duplicates().copy()
dim_sellers = dim_sellers.reset_index(drop=True)
dim_sellers.insert(0, "seller_key", dim_sellers.index + 1)

# -------------------------
# dim_geolocation
# Grain: 1 row = 1 zip code prefix
# -------------------------
dim_geo_cols = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
]

optional_geo_cols = [
    "geolocation_record_count",
]

dim_geolocation = geo[[c for c in dim_geo_cols + optional_geo_cols if c in geo.columns]].drop_duplicates().copy()
dim_geolocation = dim_geolocation.reset_index(drop=True)
dim_geolocation.insert(0, "geolocation_key", dim_geolocation.index + 1)

# -------------------------
# dim_date
# Build from all relevant dates in orders + reviews + items
# Grain: 1 row = 1 calendar date
# -------------------------
date_series = []

for col in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    if col in orders.columns:
        date_series.append(orders[col].dropna().dt.normalize())

for col in ["review_creation_date", "review_answer_timestamp"]:
    if col in reviews.columns:
        date_series.append(reviews[col].dropna().dt.normalize())

if "shipping_limit_date" in items.columns:
    date_series.append(items["shipping_limit_date"].dropna().dt.normalize())

all_dates = pd.concat(date_series).drop_duplicates().sort_values().reset_index(drop=True)

dim_date = pd.DataFrame({"full_date": all_dates})
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["weekday_num"] = dim_date["full_date"].dt.weekday
dim_date["weekday_name"] = dim_date["full_date"].dt.day_name()
dim_date["is_weekend"] = dim_date["weekday_num"].isin([5, 6]).astype(int)
dim_date = dim_date[[
    "date_key",
    "full_date",
    "year",
    "quarter",
    "month",
    "month_name",
    "day",
    "weekday_num",
    "weekday_name",
    "is_weekend",
]]

# =========================================================
# BUILD LOOKUP MAPS FOR SURROGATE KEYS
# =========================================================
customer_key_map = dim_customers[["customer_id", "customer_key"]]
product_key_map = dim_products[["product_id", "product_key"]]
seller_key_map = dim_sellers[["seller_id", "seller_key"]]
geo_key_map = dim_geolocation[["geolocation_zip_code_prefix", "geolocation_key"]]

# =========================================================
# FACT TABLE: fact_orders
# Grain: 1 row = 1 order
# =========================================================

# Base join
fact_orders = (
    orders
    .merge(customer_key_map, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments, on="order_id", how="left")
    .merge(reviews, on="order_id", how="left", suffixes=("", "_review"))
)

# Add customer geolocation key through customers
customer_geo_bridge = dim_customers[["customer_id", "customer_zip_code_prefix"]].merge(
    geo_key_map,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)[["customer_id", "geolocation_key"]].rename(columns={"geolocation_key": "customer_geolocation_key"})

fact_orders = fact_orders.merge(customer_geo_bridge, on="customer_id", how="left")

# Add date keys
def add_date_key(df, source_col, target_col):
    if source_col in df.columns:
        df[target_col] = pd.to_datetime(df[source_col], errors="coerce").dt.strftime("%Y%m%d")
        df[target_col] = pd.to_numeric(df[target_col], errors="coerce").astype("Int64")
    return df

fact_orders = add_date_key(fact_orders, "order_purchase_timestamp", "purchase_date_key")
fact_orders = add_date_key(fact_orders, "order_approved_at", "approved_date_key")
fact_orders = add_date_key(fact_orders, "order_delivered_carrier_date", "carrier_date_key")
fact_orders = add_date_key(fact_orders, "order_delivered_customer_date", "delivered_date_key")
fact_orders = add_date_key(fact_orders, "order_estimated_delivery_date", "estimated_delivery_date_key")

if "review_creation_date" in fact_orders.columns:
    fact_orders = add_date_key(fact_orders, "review_creation_date", "review_creation_date_key")
if "review_answer_timestamp" in fact_orders.columns:
    fact_orders = add_date_key(fact_orders, "review_answer_timestamp", "review_answer_date_key")

# Fill numeric aggregates after left join
numeric_fill_zero_cols = [
    "item_count",
    "item_price_total",
    "item_price_mean",
    "item_price_max",
    "freight_total",
    "freight_mean",
    "seller_count",
    "product_count",
    "order_gross_value",
    "payment_value_total",
    "payment_value_mean",
    "payment_value_max",
    "payment_installments_max",
    "payment_installments_mean",
    "payment_count",
    "payment_type_nunique",
    "flag_multi_payment",
    "has_comment",
    "review_length",
    "flag_long_review",
    "is_satisfied",
]
for col in numeric_fill_zero_cols:
    if col in fact_orders.columns:
        fact_orders[col] = fact_orders[col].fillna(0)

# Optional text fill
for col in ["payment_type_main", "review_comment_title", "review_comment_message", "review_sentiment"]:
    if col in fact_orders.columns:
        fact_orders[col] = fact_orders[col].fillna("unknown")

# Select fact_orders columns
fact_orders_cols = [
    "order_id",
    "customer_key",
    "customer_geolocation_key",

    "purchase_date_key",
    "approved_date_key",
    "carrier_date_key",
    "delivered_date_key",
    "estimated_delivery_date_key",
    "review_creation_date_key",
    "review_answer_date_key",

    "customer_id",
    "order_status",
    "order_stage",

    "is_approved",
    "is_shipped",
    "is_delivered",

    "flag_carrier_before_approval",
    "flag_customer_before_carrier",
    "flag_inconsistent_status_delivery",
    "flag_canceled_but_delivered",

    "approval_delay_hours",
    "shipping_delay_days",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days",
    "is_late_delivery",

    "item_count",
    "item_price_total",
    "item_price_mean",
    "item_price_max",
    "freight_total",
    "freight_mean",
    "seller_count",
    "product_count",
    "order_gross_value",

    "payment_value_total",
    "payment_value_mean",
    "payment_value_max",
    "payment_installments_max",
    "payment_installments_mean",
    "payment_count",
    "payment_type_main",
    "payment_type_nunique",
    "flag_multi_payment",

    "review_id",
    "review_score",
    "has_comment",
    "review_length",
    "flag_long_review",
    "review_sentiment",
    "is_satisfied",

    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

fact_orders = fact_orders[[c for c in fact_orders_cols if c in fact_orders.columns]].drop_duplicates(subset=["order_id"]).copy()

# Add surrogate fact key
fact_orders = fact_orders.reset_index(drop=True)
fact_orders.insert(0, "order_key", fact_orders.index + 1)

# =========================================================
# FACT TABLE: fact_order_items
# Grain: 1 row = 1 order item
# =========================================================
fact_order_items = (
    items
    .merge(fact_orders[["order_id", "order_key", "customer_key"]], on="order_id", how="left")
    .merge(product_key_map, on="product_id", how="left")
    .merge(seller_key_map, on="seller_id", how="left")
)

# Add shipping date key
fact_order_items = add_date_key(fact_order_items, "shipping_limit_date", "shipping_limit_date_key")

fact_order_items_cols = [
    "order_key",
    "order_id",
    "customer_key",
    "product_key",
    "seller_key",
    "order_item_id",
    "shipping_limit_date_key",
    "shipping_limit_date",
    "price",
    "freight_value",
    "total_item_value",
    "freight_ratio",
    "flag_zero_price",
    "flag_price_outlier",
    "flag_high_freight",
    "product_id",
    "seller_id",
]

fact_order_items = fact_order_items[[c for c in fact_order_items_cols if c in fact_order_items.columns]].copy()
fact_order_items = fact_order_items.reset_index(drop=True)
fact_order_items.insert(0, "order_item_key", fact_order_items.index + 1)

# =========================================================
# OPTIONAL: VALIDATION
# =========================================================
print("dim_customers:", dim_customers.shape)
print("dim_products:", dim_products.shape)
print("dim_sellers:", dim_sellers.shape)
print("dim_geolocation:", dim_geolocation.shape)
print("dim_date:", dim_date.shape)
print("fact_orders:", fact_orders.shape)
print("fact_order_items:", fact_order_items.shape)

print("Duplicate order_id in fact_orders:", fact_orders["order_id"].duplicated().sum())
print("Duplicate customer_id in dim_customers:", dim_customers["customer_id"].duplicated().sum())
print("Duplicate product_id in dim_products:", dim_products["product_id"].duplicated().sum())
print("Duplicate seller_id in dim_sellers:", dim_sellers["seller_id"].duplicated().sum())
print("Duplicate zip in dim_geolocation:", dim_geolocation["geolocation_zip_code_prefix"].duplicated().sum())

# =========================================================
# SAVE OUTPUTS
# =========================================================
dim_customers.to_csv(OUTPUT_DIR / "dim_customers.csv", index=False)
dim_products.to_csv(OUTPUT_DIR / "dim_products.csv", index=False)
dim_sellers.to_csv(OUTPUT_DIR / "dim_sellers.csv", index=False)
dim_geolocation.to_csv(OUTPUT_DIR / "dim_geolocation.csv", index=False)
dim_date.to_csv(OUTPUT_DIR / "dim_date.csv", index=False)

fact_orders.to_csv(OUTPUT_DIR / "fact_orders.csv", index=False)
fact_order_items.to_csv(OUTPUT_DIR / "fact_order_items.csv", index=False)

print(f"\nStar schema tables saved to: {OUTPUT_DIR.resolve()}")

dim_customers: (99441, 11)
dim_products: (32951, 14)
dim_sellers: (3095, 9)
dim_geolocation: (19011, 7)
dim_date: (755, 10)
fact_orders: (99441, 57)
fact_order_items: (112650, 18)
Duplicate order_id in fact_orders: 0
Duplicate customer_id in dim_customers: 0
Duplicate product_id in dim_products: 0
Duplicate seller_id in dim_sellers: 0
Duplicate zip in dim_geolocation: 0

Star schema tables saved to: /content/star_schema_output


# Generate Database File

In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import sqlite3
import os

df = pd.read_csv("sellers_clean.csv")
df.shape

(3095, 8)

In [ ]:
# Input nama database dan nama table kemudian jalankan code untuk menyimpan table dalam database tersebut.
db_name = 'Olist_Database.db'
table_name = 'sellers'

conn = sqlite3.connect(db_name)
df.to_sql(table_name, conn, if_exists='replace', index=False)
conn.close()

print(f"Dataframe successfully converted and saved to '{table_name}' table in '{db_name}'.")

# Catatan
# -  Apabila berhasil maka akan terbentuk file db_name.db
# -  Apabila mau menyimpan table lain ke dalam database yang sama, silahkan jalankan code ini lagi dengan nama table yang berbeda.

Dataframe successfully converted and saved to 'sellers' table in 'Olist_Database.db'.


In [ ]:
# Code ini untuk membuat koneksi dengan database yang ada dan melakukan query SQL
conn = sqlite3.connect(db_name)
query = "SELECT * FROM sellers"
df_products = pd.read_sql_query(query, conn)
conn.close()

print(f"Displaying the first 5 rows of the '{table_name}' table from '{db_name}':")
display(df_products.head())

Displaying the first 5 rows of the 'sellers' table from 'Olist_Database.db':


,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_city_clean,flag_invalid_state,seller_region,flag_missing_geo_info
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,campinas,0,southeast,0
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,mogi guacu,0,southeast,0
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,rio de janeiro,0,southeast,0
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,sao paulo,0,southeast,0
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,braganca paulista,0,southeast,0
